# BirdCLEF 2026 - Submission v55 - GPU-trained model_v50

Single-model submission using `model_v50.keras`, the EfficientNetB0 trained on Kaggle GPU on FULL 35k focal recordings + 1.5k labelled soundscape windows with mixup, label smoothing, SpecAugment and cosine-decay AdamW. Best epoch 13 reached local soundscape macro-AUC **0.8619** on the held-out 312-window split.

**Caveat**: the local val set has only ~1.3 windows per class so the local 0.86 may be optimistic. The LB will be more stable and likely lower. We submit this single-model first to read off the local-LB gap, then decide whether to ensemble with the multi-seed CPU models.

Inference: 5 temporal TTA offsets, geometric-mean aggregation, CPU only, no internet, ~30-40 min on Kaggle CPU.

In [ ]:
import os, glob, json, shutil, zipfile, warnings
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow import keras
warnings.filterwarnings('ignore')
print('TF', tf.__version__, 'librosa', librosa.__version__)

In [ ]:
MODEL_PATH = '/kaggle/input/datasets/danielemalerba0302/birdclef2026-model/model_v50.keras'
TEST_AUDIO_DIR = '/kaggle/input/competitions/birdclef-2026/test_soundscapes'
SAMPLE_SUB_PATH = '/kaggle/input/competitions/birdclef-2026/sample_submission.csv'
SUBMISSION_PATH = '/kaggle/working/submission.csv'

# Audio params - MUST match v50 training (same as our CPU pipeline)
SAMPLE_RATE = 32000
DURATION = 5.0
N_MELS = 64
N_FFT = 2048
HOP_LENGTH = 256
FMIN = 20
FMAX = 16000
TOP_DB = 40.0
MEL_NORM = 'slaney'
USE_HTK = True
TARGET_HEIGHT = 64
TARGET_WIDTH = 626

# TTA - 5 temporal offsets, geometric mean
TTA_OFFSETS = [-1.0, -0.5, 0.0, 0.5, 1.0]
BATCH_SIZE = 64

assert os.path.exists(MODEL_PATH), f'Missing model: {MODEL_PATH}'
assert os.path.exists(SAMPLE_SUB_PATH), f'Missing sample submission'
print('Model present')

In [ ]:
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
SPECIES_LIST = list(sample_sub.columns[1:])
assert len(SPECIES_LIST) == 234

In [ ]:
# .keras patcher (handles BatchNorm renorm keys for Keras compat)
BAD_KEYS = ['renorm', 'renorm_clipping', 'renorm_momentum', 'quantization_config']
def _strip(o):
    if isinstance(o, dict):
        for k in BAD_KEYS:
            o.pop(k, None)
        for v in o.values():
            _strip(v)
    elif isinstance(o, list):
        for it in o:
            _strip(it)

PATCHED = '/kaggle/working/v50_patched.keras'
tmp = '/kaggle/working/v50_patch_tmp'
if os.path.exists(tmp): shutil.rmtree(tmp)
os.makedirs(tmp)
with zipfile.ZipFile(MODEL_PATH, 'r') as z:
    z.extractall(tmp)
with open(os.path.join(tmp, 'config.json')) as f:
    cfg = json.load(f)
_strip(cfg)
with open(os.path.join(tmp, 'config.json'), 'w') as f:
    json.dump(cfg, f)
with zipfile.ZipFile(PATCHED, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(tmp):
        for fn in files:
            full = os.path.join(root, fn)
            z.write(full, os.path.relpath(full, tmp))

model = keras.models.load_model(PATCHED, compile=False, safe_mode=False)
print('input', model.input_shape, 'output', model.output_shape)
assert model.input_shape[1:] == (TARGET_HEIGHT, TARGET_WIDTH, 3)
assert model.output_shape[-1] == 234

In [ ]:
def audio_to_spectrogram(audio_arr, sr=SAMPLE_RATE):
    mel = librosa.feature.melspectrogram(
        y=audio_arr, sr=sr,
        n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=FMIN, fmax=FMAX, norm=MEL_NORM, htk=USE_HTK,
    )
    mel = np.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=TOP_DB)
    mel_db = np.nan_to_num(mel_db, nan=-TOP_DB, posinf=0.0, neginf=-TOP_DB)
    mel_norm = (mel_db + TOP_DB) / TOP_DB
    mel_norm = np.clip(mel_norm, 0, 1)
    spec = np.stack([mel_norm, mel_norm, mel_norm], axis=-1).astype(np.float32)
    if spec.shape[1] < TARGET_WIDTH:
        spec = np.pad(spec, ((0, 0), (0, TARGET_WIDTH - spec.shape[1]), (0, 0)), mode='constant')
    elif spec.shape[1] > TARGET_WIDTH:
        spec = spec[:, :TARGET_WIDTH, :]
    return spec

def extract_shifted_block(y, start_second, offset_second):
    block_samples = int(DURATION * SAMPLE_RATE)
    start_sample = int(round((start_second + offset_second) * SAMPLE_RATE))
    end_sample = start_sample + block_samples
    left_pad = max(0, -start_sample)
    right_pad = max(0, end_sample - len(y))
    start_sample = max(0, start_sample)
    end_sample = min(len(y), end_sample)
    block = y[start_sample:end_sample]
    if left_pad or right_pad:
        block = np.pad(block, (left_pad, right_pad), mode='constant')
    if len(block) < block_samples:
        block = np.pad(block, (0, block_samples - len(block)), mode='constant')
    elif len(block) > block_samples:
        block = block[:block_samples]
    return block

In [ ]:
audio_files = []
for ext in ['*.ogg', '*.wav', '*.flac', '*.mp3']:
    audio_files.extend(glob.glob(os.path.join(TEST_AUDIO_DIR, '**', ext), recursive=True))
audio_files = sorted(audio_files)
print('test audio files:', len(audio_files))

expected_by_file = {}
for row_id in sample_sub['row_id']:
    file_stem = '_'.join(row_id.split('_')[:-1])
    expected_by_file.setdefault(file_stem, []).append(row_id)

preds_by_row = {}
EPS = 1e-7

for i, audio_path in enumerate(audio_files):
    file_stem = os.path.splitext(os.path.basename(audio_path))[0]
    if i % 10 == 0:
        print(f'{i+1}/{len(audio_files)}: {file_stem}')
    if file_stem not in expected_by_file:
        continue
    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

    file_specs = []
    file_row_ids = []
    for row_id in expected_by_file[file_stem]:
        end_sec = int(row_id.split('_')[-1])
        start_sec = end_sec - int(DURATION)
        for off in TTA_OFFSETS:
            block = extract_shifted_block(y, start_sec, off)
            file_specs.append(audio_to_spectrogram(block, sr=SAMPLE_RATE))
            file_row_ids.append(row_id)
    if not file_specs:
        continue
    X = np.array(file_specs, dtype=np.float32)
    p = np.clip(model.predict(X, batch_size=BATCH_SIZE, verbose=0), 0, 1)
    for rid, pred in zip(file_row_ids, p):
        preds_by_row.setdefault(rid, []).append(pred)

print('rows with preds:', len(preds_by_row))

In [ ]:
expected_row_ids = list(sample_sub['row_id'])
rows = []
for rid in expected_row_ids:
    if rid in preds_by_row:
        arr = np.stack(preds_by_row[rid], axis=0)
        log_mean = np.log(np.clip(arr, EPS, 1.0)).mean(axis=0)
        rows.append(np.clip(np.exp(log_mean), 0.0, 1.0))
    else:
        rows.append(np.zeros(len(SPECIES_LIST), dtype=np.float32))
arr = np.stack(rows, axis=0).astype(np.float32) if rows else np.zeros((len(expected_row_ids), len(SPECIES_LIST)), dtype=np.float32)
submission_df = pd.DataFrame(arr, columns=SPECIES_LIST)
submission_df.insert(0, 'row_id', expected_row_ids)
submission_df.to_csv(SUBMISSION_PATH, index=False)
print('saved', SUBMISSION_PATH, submission_df.shape)
submission_df.head()

In [ ]:
check_df = pd.read_csv(SUBMISSION_PATH)
sample_check = pd.read_csv(SAMPLE_SUB_PATH)
assert list(check_df.columns) == list(sample_check.columns)
assert list(check_df['row_id']) == list(sample_check['row_id'])
vals = check_df.iloc[:, 1:].values
print('rows', len(check_df), 'cols', len(check_df.columns))
print('min', vals.min(), 'max', vals.max(), 'NaN', np.isnan(vals).any())
assert not np.isnan(vals).any()
assert vals.min() >= 0 and vals.max() <= 1
print('Submission ready.')